# Rabi AWG marker/reference test (AWG only, no other instruments)

Purpose: before building the full Rabi sweep, verify on a scope whether the `DATA:SEQ` sequence table's per-segment `marker_mode` field (e.g. `"highAtStart"`) fires **once when a `"repeat"`-type segment's whole N-times-repeated block starts**, or **once every individual repeat** (N times). This codebase has only ever used `marker_mode` on `"once"`-type segments (see `t1_test.py`'s `readout` segment, `pulsed_odmr.py`'s CH1 `readout` segment) -- never on a `"repeat"`-type segment -- so this specific behavior is unverified.

This matters because the planned Rabi lock-in reference works like this: CH1 plays the *same* laser-pulse-plus-gap "rep" arb for two back-to-back blocks of `N_REPS` repeats each, with the first block's sequence-table entry marked `"highAtStart"` and the second marked `"lowAtStart"`. If the marker only toggles once per block, CH1's Sync BNC output is a clean slow square wave -- exactly what the SR830's external reference needs. If it instead fires on every repeat, the Sync output would be a burst of N quick edges during the "on" block followed by silence during the "off" block, which is *not* a valid periodic reference for the lock-in to phase-lock to.

This notebook drives **only the AWG** (CH1 = laser pulse train + block marker, CH2 = MW gate pulse, same physical setup as `pulsed_odmr.py`) -- no generator, lock-in, PSUs, or interlock. Nothing here reads anything back electronically; you verify by probing with a real oscilloscope:

1. **CH1 analog output**: should show `N_REPS` bright laser pulses (each followed by a dark gap containing where the MW pulse would go), then `N_REPS` more identical pulses, repeating forever -- same shape both blocks, since CH1's *analog* signal doesn't change between blocks, only the marker does.
2. **CH2 analog output**: should show a real MW gate pulse inside the dark gap for the first `N_REPS` reps ("mw-on" block), then flat low for the next `N_REPS` reps ("mw-off" block).
3. **CH1's Sync/marker BNC output** (this is the one that matters): put it on the scope and check whether it's a single clean square wave at the block period, or a train of pulses. Report back what you see -- if it's multiple edges instead of one clean transition per block, the sequence-table marker approach doesn't work as planned and we'll need the fallback (a single big pre-concatenated arb per block with per-sample marker data, if that's even supported -- separate thing to check).
4. Optionally, scope CH1 and CH2 together to eyeball whether `PHASe:SYNChronize` is actually keeping them aligned (same open item flagged in `notes.md`'s pulsed-ODMR section for `pulsed_odmr.py`).

## Connect to the AWG

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import ks33600a

AWG_RESOURCE = "USB0::0x0957::0x5707::MY53800810::INSTR"

awg = ks33600a.KS33600A(AWG_RESOURCE, debug=True)

Keysight 33600A: connected
*RST => +0,"No error"
*CLS => +0,"No error"
SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"


## Parameters

Kept small/fast so the block period is easy to find on a scope. `N_REPS` is exactly the thing we're testing the marker behavior against -- start small (so you can visually count edges) and bump it up once you've confirmed the basic behavior.

`MW_US` is a fixed representative pulse width here, NOT swept -- this notebook only tests the marker/reference mechanism, not the real Rabi sweep.

In [2]:
FS = 1e9  # sample rate, same convention as t1_test.py / pulsed_odmr.py

LASER_US = 2.0   # laser pulse duration
PRE_US = 1.0     # padding before the MW pulse (settle time)
MW_US = 2.0      # MW pulse duration (tau_mw) -- fixed here, swept in the real experiment
POST_US = 1.0    # padding after the MW pulse before the next laser pulse --
                  # keeps RF and the next readout from overlapping in time,
                  # same rationale as the CW-ODMR RF-pickup fix (see notes.md)

N_REPS = 5  # reps per block -- start small, increase once marker behavior is confirmed

REP_US = LASER_US + PRE_US + MW_US + POST_US
BLOCK_US = N_REPS * REP_US
print(f"one rep = {REP_US} us, one block = {BLOCK_US} us, "
      f"full reference period = {2 * BLOCK_US} us ({1 / (2 * BLOCK_US * 1e-6):.1f} Hz)")

one rep = 6.0 us, one block = 30.0 us, full reference period = 60.0 us (16666.7 Hz)


## Build the arb waveforms

CH1's `rep` arb is used, UNCHANGED, for both blocks -- only the sequence table's `marker_mode` differs between the two entries that reference it, which is exactly the thing under test.

CH2 needs two different arbs, since its actual gate signal (not just the marker) differs between blocks: `gate_on_rep` (low, then a real MW pulse, then low) for the mw-on block, and `gate_off_rep` (low the whole time) for the mw-off block. Both must have the exact same total sample count as CH1's `rep` arb, or the two channels' blocks would drift out of step -- checked with an assert below (same lesson as `pulsed_odmr.py`'s `us_to_samples()` rounding-consistency comment).

In [3]:
def us_to_samples(duration_us):
    return max(1, round(duration_us * 1e-6 * FS))


def rf_pulse(freq_hz, n_samples):
    t = np.arange(n_samples) / FS
    return np.sin(2 * np.pi * freq_hz * t).astype(np.float32)


def const(n_samples, value):
    return np.full(n_samples, value, dtype=np.float32)


laser_samples = us_to_samples(LASER_US)
pre_samples = us_to_samples(PRE_US)
mw_samples = us_to_samples(MW_US)
post_samples = us_to_samples(POST_US)

# CH1: one bright laser pulse followed by a flat dark gap spanning where
# the MW pulse goes on CH2 -- CH1 itself doesn't care about the MW timing
# internal structure, only the total gap length.
ch1_rep = np.concatenate([
    rf_pulse(80e6, laser_samples),
    const(pre_samples + mw_samples + post_samples, 0.0),
])

# CH2, mw-on block: low during the laser pulse + pre-padding, high during
# the MW pulse, low during post-padding. +1/-1 normalized, mapped to a real
# 0-5V swing in the output-config cell below (same convention as
# pulsed_odmr.py's gate_pre/gate_high/gate_post).
ch2_gate_on_rep = np.concatenate([
    const(laser_samples + pre_samples, -1.0),
    const(mw_samples, 1.0),
    const(post_samples, -1.0),
])

# CH2, mw-off block: low the entire rep -- switch parked on the dump path
# throughout, no MW pulse at all.
ch2_gate_off_rep = const(laser_samples + pre_samples + mw_samples + post_samples, -1.0)

assert len(ch1_rep) == len(ch2_gate_on_rep) == len(ch2_gate_off_rep), (
    "CH1 and CH2 rep arbs must have identical sample counts, or the two "
    "channels' blocks will drift out of step"
)
print(f"rep length: {len(ch1_rep)} samples ({len(ch1_rep) / FS * 1e6:.3f} us)")

awg.upload_waveform(ch1_rep, arb_name="rep", ch=1, sample_rate=FS)
awg.upload_waveform(ch2_gate_on_rep, arb_name="gate_on_rep", ch=2, sample_rate=FS)
awg.upload_waveform(ch2_gate_off_rep, arb_name="gate_off_rep", ch=2, sample_rate=FS)

rep length: 6000 samples (6.000 us)
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2


## Build and upload the sequences

Same `build_block_descriptor()` as `t1_test.py`/`pulsed_odmr.py`. CH1's sequence is the marker under test: the SAME `rep` arb, split into two `N_REPS`-rep halves marked `"highAtStart"`/`"lowAtStart"`. CH2's sequence mirrors the block structure but swaps which arb plays.

**Revised after checking the actual Keysight programming manual** (33500-33600 Operating and Service Guide): the manual's own `DATA:SEQuence` examples (e.g. the `MMEMory` subsystem example) use sequences made entirely of `"repeat"`-type segments with no `"once"` anchor at all, and describe them as working correctly -- directly contradicting the "needs a `once` anchor" theory from the earlier ad hoc hardware test. That test's real failure was more likely the separately-documented "re-uploading a `DATA:SEQ` name that already exists silently does not replace it" issue (see `t1_test.py`'s `upload_sequence()` docstring), triggered by re-running the same cell twice, not a fundamental restriction on all-`"repeat"` sequences.

Dropped the `"once"` anchor -- back to two plain `"repeat"` segments per block per channel, matching the manual's validated pattern exactly. Also avoiding the anchor's side effect of putting two *adjacent* segments referencing the *same* arb name with two *different* `play_control` values (`once` then `repeat`) back to back -- something no Keysight example ever does, and an unverified structural difference worth eliminating while debugging.

Sequence names bumped to `_v2` so this re-upload doesn't collide with the earlier `_ch1`/`_ch2` sequences already sitting in this AWG session's volatile memory (re-uploading under an existing name silently does nothing -- confirmed separately, see `notes.md`).

CH2 is targeted via the `SOUR2:DATA:SEQ` prefix, same as `pulsed_odmr.py` used -- confirmed accepted without error on this firmware.

In [4]:
def build_block_descriptor(sequence_name, segments):
    """Same DATA:SEQ block-descriptor builder as t1_test.py/pulsed_odmr.py."""
    parts = [f'"{sequence_name}"']
    for arb_name, repeat_count, play_control, marker_mode, marker_point in segments:
        parts.append(
            f'"{arb_name}",{repeat_count},{play_control},{marker_mode},{marker_point}'
        )
    payload = ",".join(parts)
    payload_bytes = payload.encode("utf-8")
    payload_len = len(payload_bytes)
    n = len(str(payload_len))
    return f"#{n}{payload_len}{payload}"


SEQUENCE_NAME_CH1 = "rabi_marker_test_ch1_v2"
SEQUENCE_NAME_CH2 = "rabi_marker_test_ch2_v2"

# Plain two-"repeat"-segment blocks, no "once" anchor -- matches the
# Keysight manual's own validated DATA:SEQuence examples (e.g.
# "dc5v",2,repeat,maintain,5 in the MMEMory subsystem example), which use
# only "repeat"-type segments with no anchor at all.
block1_segments = [
    ["rep", str(N_REPS), "repeat", "lowAtStart", 10],
    ["rep", str(N_REPS), "repeat", "highAtStart", 10],
]
block1 = build_block_descriptor(SEQUENCE_NAME_CH1, block1_segments)
awg.write(f"DATA:SEQ {block1}")  # unprefixed -> channel 1, per t1_test.py's convention

block2_segments = [
    ["gate_off_rep", str(N_REPS), "repeat", "lowAtStart", 10],
    ["gate_on_rep", str(N_REPS), "repeat", "lowAtStart", 10],
]
block2 = build_block_descriptor(SEQUENCE_NAME_CH2, block2_segments)
awg.write(f"SOUR2:DATA:SEQ {block2}")

DATA:SEQ #284"rabi_marker_test_ch1_v2","rep",5,repeat,lowAtStart,10,"rep",5,repeat,highAtStart,10 => +0,"No error"
SOUR2:DATA:SEQ #3100"rabi_marker_test_ch2_v2","gate_off_rep",5,repeat,lowAtStart,10,"gate_on_rep",5,repeat,lowAtStart,10 => +0,"No error"


## Configure channel output and start

Same output configuration as `t1_test.py`/`pulsed_odmr.py`: CH1 into 50 ohm at a small Vpp (laser drive convention), CH2 into a high-impedance load with a 0-5V swing (ZYSWA switch control levels). Both self-triggered (`TRIG:SOUR IMM`) so they loop the whole two-block sequence forever once turned on. `PHASe:SYNChronize` issued once, right after both outputs are enabled.

In [18]:
# CH1
awg.write("PHASe:SYNChronize")

awg.write("OUTP1:LOAD 50")
awg.write("SOUR1:FUNC:ARB:SRAT 1e9")
awg.write(f'SOUR1:FUNC:ARB "{SEQUENCE_NAME_CH1}"')
awg.write("SOUR1:FUNC ARB")
awg.write("SOUR1:VOLT 0.632")
awg.write("OUTPUT1 ON")
awg.write("TRIG1:SOUR IMM")

# CH2
awg.write("OUTP2:LOAD INF")
awg.write("SOUR2:FUNC:ARB:SRAT 1e9")
awg.write(f'SOUR2:FUNC:ARB "{SEQUENCE_NAME_CH2}"')
awg.write("SOUR2:FUNC ARB")
awg.write("SOUR2:VOLT 5.0")
awg.write("SOUR2:VOLT:OFFS 2.5")
awg.write("OUTPUT2 ON")
awg.write("TRIG2:SOUR IMM")

print("AWG running (no PHASe:SYNChronize -- still investigating that separately).")

PHASe:SYNChronize => +0,"No error"
OUTP1:LOAD 50 => +0,"No error"
SOUR1:FUNC:ARB:SRAT 1e9 => +0,"No error"
SOUR1:FUNC:ARB "rabi_marker_test_ch1_v2" => +0,"No error"
SOUR1:FUNC ARB => +0,"No error"
SOUR1:VOLT 0.632 => +0,"No error"
OUTPUT1 ON => +0,"No error"
TRIG1:SOUR IMM => +0,"No error"
OUTP2:LOAD INF => +0,"No error"
SOUR2:FUNC:ARB:SRAT 1e9 => +0,"No error"
SOUR2:FUNC:ARB "rabi_marker_test_ch2_v2" => +0,"No error"
SOUR2:FUNC ARB => +0,"No error"
SOUR2:VOLT 5.0 => +0,"No error"
SOUR2:VOLT:OFFS 2.5 => +0,"No error"
OUTPUT2 ON => +0,"No error"
TRIG2:SOUR IMM => +0,"No error"
AWG running (no PHASe:SYNChronize -- still investigating that separately).


In [ ]:
awg.query("*OPC?")  # block until the instrument confirms it's done
print("OUTP1?", awg.query("OUTP1?"))
print("OUTP2?", awg.query("OUTP2?"))
print("SYST:ERR?", awg.query("SYST:ERR?"))

PHASe:SYNChronize => +0,"No error"
OUTP1? 1
OUTP2? 1
SYST:ERR? +0,"No error"


In [14]:
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR2:FUNC?", awg.query("SOUR2:FUNC?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("SOUR2:VOLT?", awg.query("SOUR2:VOLT?"))
print("SOUR2:VOLT:OFFS?", awg.query("SOUR2:VOLT:OFFS?"))
print("SOUR2:DATA:VOL:CAT?", awg.query("SOUR2:DATA:VOL:CAT?"))

OUTP2? 1
SOUR2:FUNC? ARB
SOUR2:FUNC:ARB? "rabi_marker_test_ch2_v2"
SOUR2:VOLT? +5.0000000000000E+00
SOUR2:VOLT:OFFS? +2.5000000000000E+00
SOUR2:DATA:VOL:CAT? "EXP_RISE","GATE_ON_REP","GATE_OFF_REP","rabi_marker_test_ch2_v2"


## What to check on the oscilloscope

- **CH1 analog out**: `N_REPS` bright pulses, then `N_REPS` more, repeating -- same shape throughout (this one should look boring/unchanging; it's the control).
- **CH2 analog out**: a real gate pulse inside the dark gap for `N_REPS` reps, then flat low for the next `N_REPS` reps, repeating.
- **CH1's Sync/marker BNC**: the one under test. Look for ONE clean edge each time the block changes (period = `2 * BLOCK_US` printed above) -- vs. a burst of `N_REPS` quick pulses during the "on" block followed by silence during the "off" block. **Report back which one you see.**
- Optional: CH1 vs CH2 together, at the start of a run and again after leaving it running a while, to eyeball whether `PHASe:SYNChronize` is holding them aligned over time.

## Diagnostic: confirm the instrument's actual state

Useful any time the scope doesn't show what's expected -- queries the AWG directly rather than trusting that the writes above "must have" taken effect. If `OUTP1?`/`OUTP2?` come back `0`, the outputs never actually turned on. If they come back `1` and everything else below looks right but the scope still shows nothing, check the sequence structure itself -- confirmed on this hardware that a `DATA:SEQ` sequence made entirely of `"repeat"`-type segments (no `"once"` segment) reports a fully healthy state here yet produces no real output; see the note above `build-sequences` for the fix already applied.

In [14]:
print("OUTP1?", awg.query("OUTP1?"))
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR1:FUNC?", awg.query("SOUR1:FUNC?"))
print("SOUR2:FUNC?", awg.query("SOUR2:FUNC?"))
print("SOUR1:FUNC:ARB?", awg.query("SOUR1:FUNC:ARB?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("TRIG1:SOUR?", awg.query("TRIG1:SOUR?"))
print("TRIG2:SOUR?", awg.query("TRIG2:SOUR?"))
print("SYST:ERR?", awg.query("SYST:ERR?"))  # anything still sitting in the
                                              # error queue that debug=True
                                              # printed but didn't stop on

OUTP1? 1
OUTP2? 1
SOUR1:FUNC? ARB
SOUR2:FUNC? ARB
SOUR1:FUNC:ARB? "rabi_marker_test_ch1"
SOUR2:FUNC:ARB? "rabi_marker_test_ch2"
TRIG1:SOUR? IMM
TRIG2:SOUR? IMM
SYST:ERR? +0,"No error"


## Stop / disconnect

Run when done probing.

In [7]:
awg.write("OUTPUT1 OFF")
awg.write("OUTPUT2 OFF")
awg.close()
print("AWG outputs off, connection closed.")

OUTPUT1 OFF => +0,"No error"
OUTPUT2 OFF => +0,"No error"
AWG outputs off, connection closed.
